# 01 · Česká republika – test MA zákona

**Cíl:** Stáhnout metriky uličních segmentů a bloků pro ~843 českých měst a mestysů,
fitovat MA zákon a provést robustní statistické testování.

**Data:**
- Populace: ČSÚ `1300722503.xlsx` (k 1. 1. 2025)
- Typ sídla: RÚIAN `UI_OBEC.csv`
- Uliční sítě: OSMnx / OpenStreetMap

**Dimenze experimentu (smyčka):**
- `network_type` ∈ {`drive`, `all`}
- agregace ∈ {mean, median}
- míra velikosti ∈ {populace_celkem}
  *(rozloha bude přidána z admin. boundary v pozdější verzi)*

> Notebook je orchestrátor. Veškerá logika v `src/urbanmal/`.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from src.urbanmal import viz
from src.urbanmal.data.census import build_municipalities, filter_urban
from src.urbanmal.metrics.segments import batch_segment_stats
from src.urbanmal.fitting import fit_loglog, fit_altmann, altmann_func

viz.set_style()

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
FIGURES = Path("../reports/figures")
PROCESSED.mkdir(exist_ok=True)

print("Prostředí OK")

---
## 1 · Načtení a průzkum dat ČSÚ / RÚIAN

In [ ]:
df_all = build_municipalities(
    RAW / "UI_OBEC.csv",
    RAW / "pocty_obyvatel/1300722503.xlsx",
)
cities = filter_urban(df_all)

print(f"Urbánních sídel celkem: {len(cities)}")
print(cities["typ_sidla"].value_counts())
print()
print(cities[["nazev", "typ_sidla", "populace_celkem"]]
      .sort_values("populace_celkem", ascending=False)
      .head(10)
      .to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
import seaborn as sns
import numpy as np
sns.histplot(np.log10(cities["populace_celkem"]), bins=40,
             color=viz.palette_seq[3], ax=ax)
ax.set_xlabel("log₁₀(populace)")
ax.set_ylabel("Počet sídel")
ax.set_title("Distribuce velikosti sídel (ČR, urbánní, n=843)")
ax.set_xticks([2, 3, 4, 5, 6])
ax.set_xticklabels(["100", "1 k", "10 k", "100 k", "1 M"])
plt.tight_layout()
plt.savefig(FIGURES / "05_cz_populace_distribuce.png", dpi=150)
plt.show()

---
## 2 · Stažení uličních sítí a výpočet metrik

⚠️ **Tato buňka stahuje data pro ~843 měst a zabere desítky minut.**
Výsledek se ukládá do `data/processed/cz_segments.csv` — pokud soubor existuje, přeskočí stahování.

In [ ]:
CACHE_DRIVE = PROCESSED / "cz_segments_drive.csv"
CACHE_ALL   = PROCESSED / "cz_segments_all.csv"

# Sestavíme OSMnx query name: "<nazev>, Czech Republic"
cities = cities.copy()
cities["osm_query"] = cities["nazev"] + ", Czech Republic"

# --- drive ---
if CACHE_DRIVE.exists():
    print("Načítám z cache:", CACHE_DRIVE)
    seg_drive = pd.read_csv(CACHE_DRIVE, dtype={"kod": str})
else:
    print("Stahuji sítě (drive)...")
    seg_drive = batch_segment_stats(cities, name_col="osm_query", network_type="drive")
    seg_drive["kod"] = cities["kod"].values
    seg_drive.to_csv(CACHE_DRIVE, index=False)
    print("Uloženo:", CACHE_DRIVE)

# --- all ---
if CACHE_ALL.exists():
    print("Načítám z cache:", CACHE_ALL)
    seg_all = pd.read_csv(CACHE_ALL, dtype={"kod": str})
else:
    print("Stahuji sítě (all)...")
    seg_all = batch_segment_stats(cities, name_col="osm_query", network_type="all")
    seg_all["kod"] = cities["kod"].values
    seg_all.to_csv(CACHE_ALL, index=False)
    print("Uloženo:", CACHE_ALL)

errors_drive = seg_drive["error"].notna().sum()
errors_all   = seg_all["error"].notna().sum()
print(f"Chyby drive: {errors_drive}/{len(seg_drive)} | all: {errors_all}/{len(seg_all)}")

---
## 3 · Smyčka přes dimenze experimentu – fitting MA zákona

In [ ]:
import numpy as np
results = []

for net_type, seg_df in [("drive", seg_drive), ("all", seg_all)]:
    valid = seg_df[seg_df["error"].isna() & seg_df["mean_m"].notna()].copy()
    valid = valid.merge(cities[["osm_query", "populace_celkem", "typ_sidla"]], on="osm_query", how="left")
    valid = valid[valid["populace_celkem"] > 0].dropna(subset=["populace_celkem"])

    x = valid["populace_celkem"].values.astype(float)

    for agg in ["mean_m", "median_m"]:
        y = valid[agg].values.astype(float)
        mask = (x > 0) & (y > 0)
        x_clean, y_clean = x[mask], y[mask]

        ll = fit_loglog(x_clean, y_clean)
        try:
            alt = fit_altmann(x_clean, y_clean)
        except Exception as e:
            alt = {"a": None, "b": None, "c": None, "r2": None, "n": len(x_clean)}

        results.append({
            "network_type": net_type,
            "aggregation":  agg,
            "n":            ll["n"],
            "loglog_b":     round(ll["b"], 4),
            "loglog_r2":    round(ll["r2"], 4),
            "loglog_p":     round(ll["p_value"], 4),
            "altmann_b":    round(alt["b"], 4) if alt["b"] is not None else None,
            "altmann_r2":   round(alt["r2"], 4) if alt["r2"] is not None else None,
        })

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

---
## 4 · Vizualizace: scatter + fit křivka (primární kombinace)

In [ ]:
# Primární: drive + median
valid = seg_drive[seg_drive["error"].isna() & seg_drive["median_m"].notna()].copy()
valid = valid.merge(cities[["osm_query", "populace_celkem", "typ_sidla"]], on="osm_query", how="left")
valid = valid[(valid["populace_celkem"] > 0) & (valid["median_m"] > 0)]

x = valid["populace_celkem"].values.astype(float)
y = valid["median_m"].values.astype(float)
hue = valid["typ_sidla"]

ll   = fit_loglog(x, y)
try:
    alt = fit_altmann(x, y)
    fit_params = alt
    fit_fn     = altmann_func
except Exception:
    fit_params = {"a": ll["a"], "b": ll["b"], "c": 0.0, "r2": ll["r2"]}
    fit_fn     = altmann_func

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

viz.plot_ma_fit(
    x, y, fit_func=fit_fn, fit_params=fit_params, hue=hue,
    x_label="Populace", y_label="Medián délky segmentu (m)",
    title="MA zákon H1 – ČR, drive, medián",
    ax=axes[0]
)

viz.plot_ma_fit_loglog(
    x, y, fit_params={"a": ll["a"], "b": ll["b"], "r2": ll["r2"]},
    title=f"MA zákon H1 – log–log  (b={ll['b']:.3f}, R²={ll['r2']:.3f})",
    ax=axes[1]
)

plt.tight_layout()
plt.savefig(FIGURES / "06_cz_ma_fit_h1_drive_median.png", dpi=150)
plt.show()

---
## 5 · Diagnostika residuí

Breusch-Pagan test na heteroskedasticitu + QQ-plot residuí log-log modelu.

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

log_x = np.log(x)
log_y = np.log(y)
X_sm  = sm.add_constant(log_x)
ols   = sm.OLS(log_y, X_sm).fit()
residuals = ols.resid

bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X_sm)
print(f"Breusch-Pagan: stat={bp_stat:.3f}, p={bp_p:.4f}")
print("OLS summary:")
print(ols.summary())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sm.qqplot(residuals, line="s", ax=axes[0])
axes[0].set_title("QQ-plot residuí (log-log OLS)")
axes[1].scatter(ols.fittedvalues, residuals, alpha=0.4, color=viz.palette_seq[3], s=20)
axes[1].axhline(0, color="crimson", linewidth=1)
axes[1].set_xlabel("Fitted values")
axes[1].set_ylabel("Residua")
axes[1].set_title("Residua vs. fitted")
plt.tight_layout()
plt.savefig(FIGURES / "07_cz_diagnostika_residui.png", dpi=150)
plt.show()

---
## 6 · Bootstrap CI + permutační test

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 1000
N_PERM = 1000

# -- Bootstrap koeficientu b --
b_boots = []
for _ in range(N_BOOT):
    idx = rng.integers(0, len(x), size=len(x))
    res = fit_loglog(x[idx], y[idx])
    b_boots.append(res["b"])

b_boots = np.array(b_boots)
ci_lo, ci_hi = np.percentile(b_boots, [2.5, 97.5])
print(f"Bootstrap b: mean={b_boots.mean():.4f}, 95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"MA zákon predikuje b < 0: {'ANO' if ci_hi < 0 else 'NE – CI přesahuje 0'}")

# -- Permutační test --
obs_b = fit_loglog(x, y)["b"]
perm_bs = []
for _ in range(N_PERM):
    y_perm = rng.permutation(y)
    perm_bs.append(fit_loglog(x, y_perm)["b"])

perm_bs = np.array(perm_bs)
p_perm = np.mean(perm_bs <= obs_b)
print(f"Permutační p-hodnota (b <= {obs_b:.4f}): {p_perm:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(b_boots, bins=50, color=viz.palette_seq[3], alpha=0.7, label="Bootstrap b")
ax.axvline(ci_lo, color="steelblue", linestyle="--", label=f"95% CI [{ci_lo:.3f}, {ci_hi:.3f}]")
ax.axvline(ci_hi, color="steelblue", linestyle="--")
ax.axvline(0, color="crimson", linewidth=1.5, label="b=0 (H₀)")
ax.set_xlabel("Koeficient b")
ax.set_title("Bootstrap distribuce koeficientu b – H1 (drive, median)")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "08_cz_bootstrap_b.png", dpi=150)
plt.show()

---
## 7 · Leave-one-out – vliv Prahy

Praha je extrémní outlier (1,4M obyvatel). Je výsledek na ní závislý?

In [ ]:
mask_no_praha = valid["nazev"].str.lower() != "praha"
x_np = valid.loc[mask_no_praha, "populace_celkem"].values.astype(float)
y_np = valid.loc[mask_no_praha, "median_m"].values.astype(float)

ll_full    = fit_loglog(x,    y)
ll_no_prah = fit_loglog(x_np, y_np)

print(f"Plný vzorek (n={ll_full['n']}):    b={ll_full['b']:.4f}, R²={ll_full['r2']:.4f}")
print(f"Bez Prahy   (n={ll_no_prah['n']}): b={ll_no_prah['b']:.4f}, R²={ll_no_prah['r2']:.4f}")

---
## 8 · Závěry a přechod na Fázi 2

*(Doplnit po spuštění)*

| Dimenze | Výsledek |
|---|---|
| H1 potvrzena? | *(b < 0, CI nezahrnuje 0?)* |
| Silnější: drive vs. all? | *(R² srovnání)* |
| Silnější: mean vs. median? | *(R² srovnání)* |
| Vliv Prahy | *(LOO výsledek)* |
| Heteroskedasticita | *(BP test p-hodnota)* |